# Notebook 03b — Gene × Motif Analysis, Class-Level (N=84)

## Why this notebook exists

Notebook 03 tested gene-motif associations at N=179 neurons, producing 2,797 FDR
survivors with a clean permutation null (median 0, 95th pctile 0). **But** CeNGEN
expression is class-level: L/R-paired neurons (ADAL, ADAR) share identical
expression vectors because they map to the same CeNGEN class (ADA). Of the 179
Witvliet neurons with expression, only 84 unique expression classes exist. The
N=179 Spearman p-values are therefore anti-conservative by a factor related to
the within-class replication.

This notebook runs the same association test at the correct level: **average
motif features per CeNGEN class, test at N=84**. The number of FDR survivors here
is the honest one to report.

## Preregistered hypotheses

- **H₁**: Gene-motif associations replicate at class level. Number of FDR
  survivors is lower than at neuron-level (because N is smaller) but still
  **≥ 10 genes** at q_global < 0.05 across all tests.
- **H₀**: After proper N-correction, no associations survive. The N=179 signal
  in Notebook 03 was real in pattern but the effect was carried by a handful of
  high-expression classes and diluted to non-significance at N=84.

## Preregistered success criteria

1. **N** = 84 ± 2 (expected exact count from Notebook 02's mapping).
2. **At least 10 genes** survive q_global < 0.05 at class level.
3. **Permutation null** (100 class-shuffles) 95th pctile < observed hits.
4. **Overlap with Notebook 03**: ≥ 50% of class-level survivors were also in
   Notebook 03's neuron-level top-200 hits. *(Real signal should be stable.)*

## Halting rule

If criterion 2 fails, the honest answer is **weak-positive**: the pattern was
real but the effect size is too small to defend at proper N. Document and proceed.

If both 2 and 3 fail: declare null, close out Paper 1's positive-finding path,
pivot to Notebook 05 (developmental rewiring) as main story.

If 2 and 3 pass: strong-positive — Paper 1 has a real finding, proceed to
biological interpretation.

In [1]:
import sys, time
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.motifs import compute_motifs
from lib.paths import DERIVED

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

RNG = np.random.default_rng(42)

# Load everything from Notebook 01 + 02's saved artifacts
adult = np.load(DERIVED / 'connectome_adult.npz', allow_pickle=True)
neurons_c = adult['neurons']
chem_adj, gap_adj = adult['chem_adj'], adult['gap_adj']

expr_data = np.load(DERIVED / 'expression_tpm.npz', allow_pickle=True)
neurons_e, tpm = expr_data['neurons'], expr_data['tpm']
genes_csv = pd.read_csv(DERIVED / 'expression_genes.csv')

# Mapping neuron -> cengen_class (from Notebook 02)
mapping = pd.read_csv(DERIVED / 'expression_neuron_mapping.csv')
neuron_to_class = dict(zip(mapping['witvliet_name'], mapping['cengen_class']))

print(f'Witvliet neurons: {len(neurons_c)}')
print(f'Mapped to CeNGEN: {mapping["cengen_class"].notna().sum()}')
print(f'Unique CeNGEN classes: {mapping["cengen_class"].nunique()}')

Witvliet neurons: 222
Mapped to CeNGEN: 179
Unique CeNGEN classes: 84


## Step 1 — Compute motif features per neuron, then aggregate per class

In [2]:
# Use the full-N=179 motif features we already computed
mf_per_neuron = pd.read_csv(DERIVED / 'motif_features.csv', index_col=0)

# Attach cengen_class to each row
mf_per_neuron['cengen_class'] = mf_per_neuron.index.map(neuron_to_class)
# Drop neurons without a class (non-neurons + ambiguous)
mf_with_class = mf_per_neuron[mf_per_neuron['cengen_class'].notna()]
print(f'Neurons with class: {len(mf_with_class)}')

# Average motif features within each class
motif_cols_to_aggregate = [c for c in mf_with_class.columns if c != 'cengen_class']
mf_per_class = mf_with_class.groupby('cengen_class')[motif_cols_to_aggregate].mean()
print(f'Classes: {mf_per_class.shape}')

# Save
mf_per_class.to_csv(DERIVED / 'motif_features_per_class.csv')

Neurons with class: 179
Classes: (84, 15)


## Step 2 — Build class-level expression matrix (one row per class)

In [3]:
# For each CeNGEN class, the expression vector is identical across all L/R
# neurons in that class (CeNGEN pooled them at source). Just take the first.
class_to_expr = {}
for i, neuron in enumerate(neurons_e):
    cls = neuron_to_class.get(str(neuron))
    if isinstance(cls, str) and cls not in class_to_expr:
        if not np.all(np.isnan(tpm[i])):
            class_to_expr[cls] = tpm[i]

# Restrict both tables to common classes, aligned
common_classes = sorted(set(class_to_expr) & set(mf_per_class.index))
print(f'Classes in both expression + motifs: {len(common_classes)}')

tpm_class = np.stack([class_to_expr[c] for c in common_classes])
mf_class_aligned = mf_per_class.loc[common_classes]
print(f'tpm_class: {tpm_class.shape}  mf_class: {mf_class_aligned.shape}')

Classes in both expression + motifs: 84
tpm_class: (84, 13669)  mf_class: (84, 15)


## Step 3 — Variance filter and Spearman tests at class level

In [4]:
# Same filter rule as Notebook 03, applied to class-level matrix
MIN_CLASSES_EXPRESSED = 5   # expressed in >= 5 of 84 classes (was 10 of 179)
MIN_CV = 1.0

means = np.nanmean(tpm_class, axis=0)
stds = np.nanstd(tpm_class, axis=0)
cv = np.where(means > 0, stds / (means + 1e-9), 0.0)
n_expressed = (tpm_class > 0).sum(axis=0)

keep_mask = (n_expressed >= MIN_CLASSES_EXPRESSED) & (cv >= MIN_CV)
gene_keep = np.where(keep_mask)[0]
tpm_kept = tpm_class[:, gene_keep]
wbg_kept = expr_data['genes_wbg'][gene_keep]
sym_kept = genes_csv['symbol'].values[gene_keep]
print(f'Genes after filter: {tpm_kept.shape[1]}')

MOTIF_TARGETS = ['ffl_resid','cycle3_resid','recip_resid','two_in_resid','two_out_resid','clust_resid','total_deg']
G, M = tpm_kept.shape[1], len(MOTIF_TARGETS)
print(f'Running {G} x {M} = {G*M} Spearman tests at class level (N={len(common_classes)})...')

Genes after filter: 8846
Running 8846 x 7 = 61922 Spearman tests at class level (N=84)...


In [5]:
rows = []
t0 = time.time()
for gi in range(G):
    x = tpm_kept[:, gi]
    for target in MOTIF_TARGETS:
        y = mf_class_aligned[target].values.astype(float)
        r, p = spearmanr(x, y, nan_policy='omit')
        rows.append((wbg_kept[gi], sym_kept[gi], target, float(r), float(p)))
results = pd.DataFrame(rows, columns=['wbgene','symbol','motif','rho','p'])
print(f'done in {time.time()-t0:.1f}s')

results['q_global'] = multipletests(results['p'].fillna(1.0).values, method='fdr_bh')[1]
results['q_per_motif'] = np.nan
for m in results['motif'].unique():
    mask = results['motif'] == m
    results.loc[mask, 'q_per_motif'] = multipletests(results.loc[mask, 'p'].fillna(1.0).values, method='fdr_bh')[1]
results = results.sort_values('p').reset_index(drop=True)
results.to_csv(DERIVED / 'gene_motif_spearman_classlevel.csv', index=False)

done in 15.3s


In [6]:
surv_global = results[results['q_global'] < 0.05]
surv_per_motif = results[results['q_per_motif'] < 0.05]

print(f'SURVIVORS at q_global<0.05 (class-level):    {len(surv_global)}')
print(f'SURVIVORS at q_per_motif<0.05 (class-level): {len(surv_per_motif)}')
print(f'\nBy motif (q_global<0.05):')
print(surv_global.groupby('motif').size().to_string())
print(f'\nTop 15 class-level hits:')
print(results.head(15).to_string())

SURVIVORS at q_global<0.05 (class-level):    0
SURVIVORS at q_per_motif<0.05 (class-level): 3

By motif (q_global<0.05):
Series([], )

Top 15 class-level hits:
            wbgene     symbol         motif       rho         p  q_global  q_per_motif
0   WBGene00009949    F53A2.1     total_deg  0.473788  0.000005  0.157859     0.022551
1   WBGene00018701     pccb-1     total_deg  0.470232  0.000006  0.157859     0.022551
2   WBGene00019249   H28G03.1     total_deg  0.466700  0.000008  0.157859     0.022551
3   WBGene00022534     syx-16  two_in_resid  0.458650  0.000011  0.177407     0.071329
4   WBGene00019458    K06H7.7  two_in_resid  0.438391  0.000030  0.299582     0.071329
5   WBGene00000235      baf-1  two_in_resid  0.436392  0.000033  0.299582     0.071329
6   WBGene00014203   ZK1058.3   recip_resid -0.434039  0.000037  0.299582     0.328227
7   WBGene00008582    F08G5.3  two_in_resid  0.430958  0.000043  0.299582     0.071329
8   WBGene00011268     pde-12  two_in_resid  0.428731  0.

## Step 4 — Class-level permutation null (100 shuffles)

In [7]:
N_PERM = 100
print(f'Running {N_PERM} class-level permutations...')
t0 = time.time()
null_hits = []
for perm_i in range(N_PERM):
    perm_idx = RNG.permutation(tpm_kept.shape[0])
    tpm_perm = tpm_kept[perm_idx]
    pvals = np.zeros(G * M)
    k = 0
    for target in MOTIF_TARGETS:
        y = mf_class_aligned[target].values.astype(float)
        yr = pd.Series(y).rank().values
        for gi in range(G):
            xr = pd.Series(tpm_perm[:, gi]).rank().values
            if np.std(xr) > 0 and np.std(yr) > 0:
                r = np.corrcoef(xr, yr)[0, 1]
                from scipy.stats import t as tdist
                n = len(xr); t = r * np.sqrt(n-2) / np.sqrt(max(1e-12, 1 - r*r))
                p = 2 * tdist.sf(abs(t), df=n-2)
            else:
                p = 1.0
            pvals[k] = p; k += 1
    q_all = multipletests(pvals, method='fdr_bh')[1]
    null_hits.append(int((q_all < 0.05).sum()))
    if (perm_i+1) % 20 == 0:
        print(f'  {perm_i+1}/{N_PERM} done, recent nulls: {null_hits[-5:]}')

null_hits = np.array(null_hits)
pd.Series(null_hits, name='n_fdr_hits').to_csv(DERIVED / 'gene_motif_permutation_null_classlevel.csv', index=False)

print(f'\ndone in {time.time()-t0:.1f}s')
print(f'class-level permutation null:')
print(f'  mean: {null_hits.mean():.2f}')
print(f'  median: {np.median(null_hits):.1f}')
print(f'  95th pctile: {np.percentile(null_hits, 95):.1f}')
print(f'  max: {null_hits.max()}')
print(f'\nObserved real hits (class-level q_global<0.05): {len(surv_global)}')

Running 100 class-level permutations...


  20/100 done, recent nulls: [0, 0, 0, 0, 0]


  40/100 done, recent nulls: [0, 0, 0, 0, 0]


  60/100 done, recent nulls: [0, 0, 0, 0, 0]


  80/100 done, recent nulls: [1, 0, 0, 0, 0]


  100/100 done, recent nulls: [0, 0, 0, 0, 0]

done in 1142.8s
class-level permutation null:
  mean: 0.03
  median: 0.0
  95th pctile: 0.0
  max: 2

Observed real hits (class-level q_global<0.05): 0


## Step 5 — Criteria check and cross-check with Notebook 03

In [8]:
N = len(common_classes)
n_hits = len(surv_global)
null_95 = float(np.percentile(null_hits, 95))

# Load Notebook 03 top-200 hits to compute overlap
nb03 = pd.read_csv(DERIVED / 'gene_motif_spearman.csv')
nb03_top = nb03.sort_values('p').head(200)
nb03_pairs = set(zip(nb03_top['wbgene'], nb03_top['motif']))

class_pairs = set(zip(surv_global['wbgene'], surv_global['motif']))
overlap = class_pairs & nb03_pairs
overlap_frac = len(overlap) / max(len(class_pairs), 1) if len(class_pairs) > 0 else 0.0

checks = [
    ('1 N in [82, 86]',                        82 <= N <= 86,               f'N={N}'),
    ('2 FDR survivors >= 10',                  n_hits >= 10,                f'{n_hits} hits'),
    ('3 observed > null 95th pctile',          n_hits > null_95,            f'real={n_hits} vs null_95={null_95:.1f}'),
    ('4 overlap with nb03-top-200 >= 50%',     overlap_frac >= 0.50,        f'{len(overlap)}/{len(class_pairs)} = {overlap_frac:.1%}'),
]

print('CLASS-LEVEL PREREGISTERED CRITERIA')
print('=' * 64)
all_pass = True
for label, passed, detail in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {label:42s}  {detail}')
    if not passed: all_pass = False
print('=' * 64)
print(f'ALL CRITERIA PASS: {all_pass}')

if n_hits >= 10 and n_hits > null_95:
    verdict = 'STRONG-POSITIVE (class-level survivors exist and exceed null)'
elif n_hits >= 1:
    verdict = 'WEAK-POSITIVE (some class-level signal, smaller than neuron-level)'
else:
    verdict = 'NULL (class-level: the 2,797 neuron-level hits were pseudoreplication artifact)'
print(f'\nVERDICT: {verdict}')

surv_global.to_csv(DERIVED / 'gene_motif_fdr_survivors_classlevel.csv', index=False)

CLASS-LEVEL PREREGISTERED CRITERIA
  [PASS] 1 N in [82, 86]                             N=84
  [FAIL] 2 FDR survivors >= 10                       0 hits
  [FAIL] 3 observed > null 95th pctile               real=0 vs null_95=0.0
  [FAIL] 4 overlap with nb03-top-200 >= 50%          0/0 = 0.0%
ALL CRITERIA PASS: False

VERDICT: NULL (class-level: the 2,797 neuron-level hits were pseudoreplication artifact)


In [9]:
# Summary stats for the writeup
print('Comparison — neuron-level (03) vs class-level (03b):')
print(f'  N_samples:       179 vs {N}')
print(f'  FDR survivors:   2797 vs {n_hits}')
print(f'  Null 95 pctile:  0.0 vs {null_95:.1f}')
print(f'  Overlap of class survivors with nb03 top 200: {overlap_frac:.1%}')

Comparison — neuron-level (03) vs class-level (03b):
  N_samples:       179 vs 84
  FDR survivors:   2797 vs 0
  Null 95 pctile:  0.0 vs 0.0
  Overlap of class survivors with nb03 top 200: 0.0%
